# 05 Answer check

Runs the RAG pipeline end to end (hybrid retrieval -> open LLM on Vertex AI -> answer with citations) on the 10 trial questions in `docs/eval/trial_in_scope_10.csv`.

Prerequisites: index built (`python -m chunky.index`), `gcloud auth application-default login` done, `config/rag.yaml` -> `generation` filled in.

In [ ]:
import csv
import json
import re
from datetime import date

from chunky.answer import answer
from chunky.index import ROOT, Embedder, load_config, load_index

cfg = load_config()
emb = cfg["embedding"]
embedder = Embedder(emb["model"], emb["revision"], emb["max_tokens"])
index = load_index()

with open(ROOT / "docs/eval/trial_in_scope_10.csv", encoding="utf-8-sig", newline="") as f:
    questions = list(csv.DictReader(f))
ref = re.compile(r"(KTK|KTY|KURS|SAĞLIK|MUAYENE|SKUTER) Art\. (\d+)")
print(len(questions), "questions | model:", cfg["generation"]["model"], "| prompt:", cfg["generation"]["prompt"])

## Single question (S03)

In [ ]:
r = answer(questions[1]["Question"], embedder, index, cfg)
print(r["answer"])
print("sources:", r["sources"])
print("unknown citations:", r["unknown_citations"])

## All 10 questions

Answers are saved with expected articles and reference answers to `docs/eval/runs/` (evidence for the report).

In [ ]:
out = ROOT / "docs/eval/runs" / f"{date.today()}_trial10_{cfg['generation']['model'].split('/')[-1]}.jsonl"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", encoding="utf-8") as f:
    for q in questions:
        r = answer(q["Question"], embedder, index, cfg)
        expected = sorted({f"{d}-m{n}" for d, n in ref.findall(q["Legal Article Reference"])})
        r.update(question_id=q["Question ID"], expected=expected, reference_answer=q["Answer"])
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"=== {q['Question ID']}: {q['Question']}\n{r['answer']}")
        print("cited:", r["cited"], "| expected:", expected, "| unknown:", r["unknown_citations"], "\n")
print("saved:", out)

## Export to Excel

One row per question, one column per field (sources, seconds, expected vs retrieved vs cited articles). Saved next to the JSONL file.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font

rows = [json.loads(line) for line in out.read_text(encoding="utf-8").splitlines()]
wb = Workbook()
ws = wb.active
ws.title = "answers"
ws.append(["Question ID", "Question", "Answer", "Sources", "Seconds", "Expected IDs", "Expected retrieved",
           "Expected cited", "Cited IDs", "Unknown citations", "Reference answer", "Model", "Prompt"])
for r in rows:
    expected = set(r["expected"])
    ws.append([
        r["question_id"], r["question"], r["answer"], "\n".join(r["sources"]), r["seconds"],
        ", ".join(r["expected"]),
        f"{len(expected & set(r['retrieved']))}/{len(expected)}",  # retrieval: expected articles among the top_k
        f"{len(expected & set(r['cited']))}/{len(expected)}",      # generation: expected articles the answer cites
        ", ".join(r["cited"]), ", ".join(r["unknown_citations"]), r["reference_answer"], r["model"], r["prompt"],
    ])

for cell in ws[1]:
    cell.font = Font(bold=True)
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
for col, width in zip("ABCDEFGHIJKLM", [10, 40, 70, 50, 9, 25, 10, 10, 25, 15, 70, 25, 10]):
    ws.column_dimensions[col].width = width
ws.freeze_panes = "A2"

xlsx = out.with_suffix(".xlsx")
wb.save(xlsx)
print("saved:", xlsx)

## Export in eval.py format

Converts this run to the results format of the team's `eval.py` (the same script scores the long-context baseline), so both systems are scored with the same metrics and judge.

Limitations: citations are article-level only (paragraph metrics will be 0); the system produces no scope flag (`pred_in_scope` is null), so only in-scope questions are comparable.

In [ ]:
def to_eval_citation(chunk_id):
    """KTK-m48 -> "KTK 48" (eval.py citation format). Other units (ekm, gm, cetvel, ...) are kept as they are."""
    doc, rest = chunk_id.split("-", 1)
    m = re.fullmatch(r"m(\d+[A-Z]?)", rest)
    return f"{doc} {m[1]}" if m else chunk_id

run_name = f"rag_{cfg['generation']['model'].split('/')[-1]}_{cfg['generation']['prompt']}"
eval_out = out.with_name(out.stem + "_eval.jsonl")
with open(eval_out, "w", encoding="utf-8") as f:
    for line in out.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        f.write(json.dumps({
            "run": run_name,
            "id": r["question_id"],
            "answer": r["answer"],
            "citations": [to_eval_citation(c) for c in r["cited"]],  # all cited IDs, including unknown ones
            "parse_ok": True,          # plain-text answers, nothing to parse
            "latency_s": r["seconds"],
            "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"],
            "pred_in_scope": None,     # no scope flag in this system yet
        }, ensure_ascii=False) + "\n")
print("saved:", eval_out)

## RAG metrics

Recall@k (retrieval), supporting article (citation recall/precision), invented citations and response time, computed from this run's file. Correct answers come from eval.py's LLM judge; declined out-of-scope questions are not measured yet.

In [ ]:
import numpy as np

# RAG metrics computed from this run's file (no LLM judge needed).
# Retrieval metrics follow the design document: Recall@k = share of questions with at least one supporting
# article in the top k (hit rate); article recall@k, Precision@k and MRR are reported next to it.
rows = [json.loads(line) for line in out.read_text(encoding="utf-8").splitlines()]
k = cfg["retrieval"]["top_k"]

def ratio(hit, total):
    return hit / total if total else 0.0

def reciprocal_rank(ranked, expected):
    return next((1 / rank for rank, cid in enumerate(ranked, 1) if cid in expected), 0.0)

per_question = []
for r in rows:
    expected, cited = set(r["expected"]), set(r["cited"])
    q = {"id": r["question_id"], "expected": len(expected)}
    for n in (5, k):
        top = r["retrieved"][:n]  # retrieved is ranked by score
        q[f"hit@{n}"] = float(bool(expected & set(top)))
        q[f"recall@{n}"] = ratio(len(expected & set(top)), len(expected))
        q[f"precision@{n}"] = len(expected & set(top)) / n
    q["rr"] = reciprocal_rank(r["retrieved"][:k], expected)            # for MRR@k
    q["citation_recall"] = ratio(len(expected & cited), len(expected))  # supporting article: expected articles cited
    q["citation_precision"] = ratio(len(expected & cited), len(cited))  # cited articles that are expected
    q["unknown_citations"] = len(r["unknown_citations"])                # cited but not retrieved (invented)
    q["seconds"] = r["seconds"]
    per_question.append(q)

print(f"{'id':5}{'exp':>5}{f'hit@{k}':>7}{f'R@{k}':>7}{'RR':>6}{'citR':>7}{'citP':>7}{'unk':>5}{'sec':>7}")
for q in per_question:
    print(f"{q['id']:5}{q['expected']:>5}{q[f'hit@{k}']:>7.0f}{q[f'recall@{k}']:>7.2f}{q['rr']:>6.2f}"
          f"{q['citation_recall']:>7.2f}{q['citation_precision']:>7.2f}{q['unknown_citations']:>5}{q['seconds']:>7.1f}")

def mean(key):  # mean over questions (same averaging as eval.py)
    return round(float(np.mean([q[key] for q in per_question])), 3)

seconds = [r["seconds"] for r in rows]
tokens_in = [r["input_tokens"] for r in rows if r.get("input_tokens") is not None]
tokens_out = [r["output_tokens"] for r in rows if r.get("output_tokens") is not None]
summary = {
    "run_file": out.name,
    "model": cfg["generation"]["model"],
    "prompt": cfg["generation"]["prompt"],
    "top_k": k,
    "sparse_weight": cfg["retrieval"]["sparse_weight"],
    "questions": len(rows),
    "retrieval": {
        f"recall@{n}_hit_rate": mean(f"hit@{n}") for n in (5, k)
    } | {
        f"article_recall@{n}": mean(f"recall@{n}") for n in (5, k)
    } | {
        f"precision@{k}": mean(f"precision@{k}"),
        f"mrr@{k}": mean("rr"),
    },
    "generation": {
        "citation_recall": mean("citation_recall"),
        "citation_precision": mean("citation_precision"),
        "answers_citing_an_expected_article": sum(q["citation_recall"] > 0 for q in per_question),
        "unknown_citations_total": sum(q["unknown_citations"] for q in per_question),
    },
    "seconds": {"mean": round(float(np.mean(seconds)), 1), "median": round(float(np.median(seconds)), 1),
                "p95": round(float(np.percentile(seconds, 95)), 1), "max": max(seconds)},
    "tokens_mean": {"input": round(float(np.mean(tokens_in))) if tokens_in else None,
                    "output": round(float(np.mean(tokens_out))) if tokens_out else None},
    "not_computed_here": ["correct answers (eval.py LLM judge)", "declined out-of-scope questions (not run yet)"],
}
print()
print(json.dumps(summary, indent=2, ensure_ascii=False))

summary_out = out.with_name(out.stem + "_summary.json")
summary_out.write_text(json.dumps({"summary": summary, "per_question": per_question}, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved:", summary_out)